<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">وزن واقعاً تغییر کرد؟</h1>
<p style="text-align:right">درس 53 از 92 · حلقهٔ آموزش را خودمان بنویسیم · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">47-loop</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/47-loop.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">یک گام آموزش را بنویسید و تغییر وزن را از صرف محاسبهٔ <bdi dir="ltr">Gradient</bdi> جدا کنید.</p><p style="text-align:right">پیش‌نیاز: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">forward</code>، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">backward</code> و <bdi dir="ltr">Optimizer</bdi>؛ ورودی و <bdi dir="ltr">Target</bdi> با شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,T)</code>.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۶۰–۱۰۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">backward</code> اجرا شود ولی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">step</code> اجرا نشود، کدام عددها تغییر می‌کنند؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
def make_model():
    torch.manual_seed(7)
    return MiniGPT(ModelConfig(12, 8, 16, 2, 1, 0.0))
x = torch.tensor([[1,2,3,4]])
y = torch.tensor([[2,3,4,5]])
model = make_model()
print('input / targets:', x.tolist(), y.tolist())
print('initial loss:', model(x,y)[1].item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">update_step(model, optimizer, x, y)</code> را بنویسید: حالت آموزش، پاک‌کردن <bdi dir="ltr">Gradient</bdi>های قبلی، محاسبهٔ <bdi dir="ltr">Loss</bdi>، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">backward</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">step</code>. مقدار <bdi dir="ltr">Loss</bdi> پیش از تغییر وزن را به‌صورت <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">float</code> برگردانید.</p>
</div>

In [ ]:
def update_step(model, optimizer, x, y):
    # TODO: یک گام آموزش
    return None

In [ ]:
def test_exercise():
    candidate = make_model()
    optimizer = torch.optim.AdamW(candidate.parameters(), lr=0.01)
    before = candidate.token_embedding.weight.detach().clone()
    result = update_step(candidate, optimizer, x, y)
    if result is None:
        return False
    assert isinstance(result, float) and result > 0
    assert not torch.equal(before, candidate.token_embedding.weight)
    assert candidate.training
    reference = make_model()
    opt = torch.optim.AdamW(reference.parameters(), lr=0.01)
    for _ in range(2):
        opt.zero_grad(set_to_none=True)
        reference(x,y)[1].backward()
        opt.step()
    update_step(candidate, optimizer, x, y)
    for expected, actual in zip(reference.parameters(), candidate.parameters()):
        torch.testing.assert_close(actual, expected)
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: update_step')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط نرخ را عوض کنید. وزن آغازین، ورودی و <bdi dir="ltr">Gradient</bdi> یکسان بمانند؛ آیا اندازهٔ تغییر هم یکسان است؟</p>
</div>

In [ ]:
for rate in (0.001, 0.01):
    trial = make_model()
    optimizer = torch.optim.SGD(trial.parameters(), lr=rate)
    before = trial.token_embedding.weight.detach().clone()
    trial(x,y)[1].backward()
    optimizer.step()
    print(rate, (trial.token_embedding.weight.detach()-before).norm().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">دو بار <bdi dir="ltr">Gradient</bdi> یک عبارت را محاسبه کرده‌ایم، بی‌آنکه <bdi dir="ltr">Gradient</bdi> قبلی پاک شود. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">gradient_once(parameter)</code> را اصلاح کنید: مشتق <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(parameter-3)**2</code> را مستقل از فراخوانی‌های قبلی برگرداند و وزن را تغییر ندهد.</p>
</div>

In [ ]:
w = torch.nn.Parameter(torch.tensor(1.0))
for _ in range(2):
    ((w-3)**2).backward()
print('accumulated gradient:', w.grad.item(), 'single gradient:', -4.0)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def gradient_once(parameter):
    # TODO: مشتق قبلی نباید جمع شود
    return None

In [ ]:
def test_repair():
    w = torch.nn.Parameter(torch.tensor(1.0))
    result = gradient_once(w)
    if result is None:
        return False
    assert float(result) == -4.0
    assert float(gradient_once(w)) == -4.0
    assert w.item() == 1.0
    assert float(gradient_once(torch.nn.Parameter(torch.tensor(4.0)))) == 2.0
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: gradient_once')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">مدل، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> واقعی است؛ حلقهٔ کوتاه این دفتر همان عملیات مرکزی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">mini_gpt/train.py</code> را بدون گزارش و ذخیره اجرا می‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">کدام آزمون نشان داد که دو گام مستقل نوشته‌اید، نه دو <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">backward</code> با <bdi dir="ltr">Gradient</bdi> انباشته؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-01/47-loop.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/47-loop.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>